# Milestone 1 Notebook: EDA and Hypothesis Testing

This notebook combines the exploratory data analysis and hypothesis testing parts of the project.

**Project topic:** Factors affecting student academic performance  
**Dataset used:** `student_performance_value.csv`  
**Target variable:** `output_grade`


## 1. Import libraries and load the dataset

In this notebook, I use pandas for data handling, matplotlib and seaborn for plots, and scipy for statistical tests.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, spearmanr

# Make plots easier to read
plt.rcParams["figure.figsize"] = (10, 6)
sns.set_theme(style="whitegrid")

# Load the dataset
df = pd.read_csv("data/student_performance_value.csv")

# Show the first rows
df.head()

## 2. Basic information about the dataset

This part checks the size of the dataset, the column names, data types, and missing values.


In [ ]:
print("Shape of the dataset:", df.shape)
print("\nColumns:")
for col in df.columns:
    print("-", col)

print("\nData types:")
print(df.dtypes)

print("\nMissing values in each column:")
print(df.isnull().sum())

## 3. Summary of the target variable

The main target variable is `output_grade`. I first inspect its distribution to understand how student performance is spread across grade categories.


In [ ]:
grade_order = ["fail", "dd", "dc", "cc", "cb", "bb", "ba", "aa"]

# Standardize grade text
df["output_grade_clean"] = df["output_grade"].astype(str).str.strip().str.lower()

grade_counts = df["output_grade_clean"].value_counts().reindex(grade_order).fillna(0)
print(grade_counts)

plt.figure(figsize=(10, 5))
sns.countplot(data=df, x="output_grade_clean", order=grade_order)
plt.title("Distribution of Output Grade")
plt.xlabel("Output Grade")
plt.ylabel("Count")
plt.show()

## 4. EDA for important explanatory variables

I focus on variables that are likely to affect academic performance:
- `weekly_study_hours`
- `attendance_to_classes`
- `taking_notes_in_classes`
- `cumulative_grade_point_average_in_the_last_semester`

These variables are directly related to study behavior, class engagement, and previous academic level.


In [ ]:
important_columns = [
    "weekly_study_hours",
    "attendance_to_classes",
    "taking_notes_in_classes",
    "cumulative_grade_point_average_in_the_last_semester"
]

for col in important_columns:
    print(f"\nValue counts for {col}:")
    print(df[col].value_counts(dropna=False))

### 4.1 Weekly study hours vs output grade

This plot helps me see whether students who study more tend to get better grades.


In [ ]:
study_order = ["none", "<5 hours", "6-10 hours", "11-20 hours", "more than 20 hours"]

plt.figure(figsize=(12, 6))
study_grade_table = pd.crosstab(df["weekly_study_hours"], df["output_grade_clean"])
study_grade_table = study_grade_table.reindex(study_order, fill_value=0)
study_grade_table = study_grade_table.reindex(columns=grade_order, fill_value=0)
study_grade_table.plot(kind="bar", stacked=True, figsize=(12, 6))
plt.title("Weekly Study Hours vs Output Grade")
plt.xlabel("Weekly Study Hours")
plt.ylabel("Count")
plt.legend(title="Output Grade", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

### 4.2 Attendance to classes vs output grade

This plot checks whether regular attendance is connected to better performance.


In [ ]:
attendance_order = ["never", "sometimes", "always"]

attendance_grade_table = pd.crosstab(df["attendance_to_classes"], df["output_grade_clean"])
attendance_grade_table = attendance_grade_table.reindex(attendance_order, fill_value=0)
attendance_grade_table = attendance_grade_table.reindex(columns=grade_order, fill_value=0)
attendance_grade_table.plot(kind="bar", stacked=True, figsize=(10, 6))
plt.title("Attendance to Classes vs Output Grade")
plt.xlabel("Attendance to Classes")
plt.ylabel("Count")
plt.legend(title="Output Grade", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

### 4.3 Taking notes in classes vs output grade

This variable is about classroom engagement. I want to see whether note-taking frequency changes grade distribution.


In [ ]:
notes_order = ["never", "sometimes", "always"]

notes_grade_table = pd.crosstab(df["taking_notes_in_classes"], df["output_grade_clean"])
notes_grade_table = notes_grade_table.reindex(notes_order, fill_value=0)
notes_grade_table = notes_grade_table.reindex(columns=grade_order, fill_value=0)
notes_grade_table.plot(kind="bar", stacked=True, figsize=(10, 6))
plt.title("Taking Notes in Classes vs Output Grade")
plt.xlabel("Taking Notes in Classes")
plt.ylabel("Count")
plt.legend(title="Output Grade", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 5. Prepare ordinal mappings for hypothesis testing

Some variables are ordered categories. To test monotonic relationships, I convert them into ordinal numeric values. This does not mean the categories are perfectly continuous, but it is useful for a simple milestone-level analysis.


In [ ]:
grade_map = {
    "fail": 0,
    "dd": 1,
    "dc": 2,
    "cc": 3,
    "cb": 4,
    "bb": 5,
    "ba": 6,
    "aa": 7
}

study_map = {
    "none": 0,
    "<5 hours": 1,
    "6-10 hours": 2,
    "11-20 hours": 3,
    "more than 20 hours": 4
}

attendance_map = {
    "never": 0,
    "sometimes": 1,
    "always": 2
}

notes_map = {
    "never": 0,
    "sometimes": 1,
    "always": 2
}

gpa_last_map = {
    "<2.00": 0,
    "2.00-2.49": 1,
    "2.50-2.99": 2,
    "3.00-3.49": 3,
    "above 3.49": 4
}

df["grade_num"] = df["output_grade_clean"].map(grade_map)
df["study_num"] = df["weekly_study_hours"].map(study_map)
df["attendance_num"] = df["attendance_to_classes"].map(attendance_map)
df["notes_num"] = df["taking_notes_in_classes"].map(notes_map)
df["last_gpa_num"] = df["cumulative_grade_point_average_in_the_last_semester"].map(gpa_last_map)

df[["output_grade_clean", "grade_num", "weekly_study_hours", "study_num"]].head()

## 6. Hypothesis tests

In this section, I test whether some patterns observed in the EDA are statistically meaningful.

### Hypothesis 1
**H0:** Weekly study hours and output grade are independent.  
**H1:** Weekly study hours and output grade are associated.

I use a **Chi-square test of independence** because both variables are categorical.


In [ ]:
contingency_study = pd.crosstab(df["weekly_study_hours"], df["output_grade_clean"])
chi2_1, p_1, dof_1, expected_1 = chi2_contingency(contingency_study)

print("Hypothesis 1: Weekly study hours vs output grade")
print("Chi-square statistic:", round(chi2_1, 4))
print("p-value:", round(p_1, 6))
print("Degrees of freedom:", dof_1)

if p_1 < 0.05:
    print("Result: Reject H0. There is a statistically significant association.")
else:
    print("Result: Fail to reject H0. No statistically significant association was found.")

### Hypothesis 2
**H0:** Attendance to classes and output grade are independent.  
**H1:** Attendance to classes and output grade are associated.

Again, I use a **Chi-square test of independence** because both variables are categorical.


In [ ]:
contingency_attendance = pd.crosstab(df["attendance_to_classes"], df["output_grade_clean"])
chi2_2, p_2, dof_2, expected_2 = chi2_contingency(contingency_attendance)

print("Hypothesis 2: Attendance to classes vs output grade")
print("Chi-square statistic:", round(chi2_2, 4))
print("p-value:", round(p_2, 6))
print("Degrees of freedom:", dof_2)

if p_2 < 0.05:
    print("Result: Reject H0. There is a statistically significant association.")
else:
    print("Result: Fail to reject H0. No statistically significant association was found.")

### Hypothesis 3
**H0:** There is no monotonic relationship between previous semester GPA and output grade.  
**H1:** There is a monotonic relationship between previous semester GPA and output grade.

Here I use **Spearman rank correlation** because both variables are ordinal after mapping.


In [ ]:
valid_gpa = df[["last_gpa_num", "grade_num"]].dropna()
rho_3, p_3 = spearmanr(valid_gpa["last_gpa_num"], valid_gpa["grade_num"])

print("Hypothesis 3: Last semester GPA vs output grade")
print("Spearman correlation:", round(rho_3, 4))
print("p-value:", round(p_3, 6))

if p_3 < 0.05:
    print("Result: Reject H0. There is a statistically significant monotonic relationship.")
else:
    print("Result: Fail to reject H0. No statistically significant monotonic relationship was found.")

## 7. Short conclusions

Based on the EDA and hypothesis tests, this milestone suggests that student performance is related to academic behavior variables such as study time, attendance, and previous GPA. These findings are useful for the next milestone, where machine learning methods can be applied to predict `output_grade`.


## 8. Notes for the next milestone

For the ML stage, the next steps can include:
- cleaning and encoding the categorical variables,
- building baseline classification models,
- comparing model performance,
- and analyzing feature importance.
